In [1]:
import numpy as np
import pandas as pd
import re
import os
from sklearn.model_selection import train_test_split

DATA_DIR = '/kaggle/input/datasets/suvroo/amazon-ml'
IMAGES_DIR = f'{DATA_DIR}/train/images'

train_raw = pd.read_csv(f'{DATA_DIR}/train.csv')
official_test_raw = pd.read_csv(f'{DATA_DIR}/test.csv')

print("train_raw:", train_raw.shape)
print("official_test_raw:", official_test_raw.shape)


train_raw: (75000, 4)
official_test_raw: (75000, 3)


In [2]:
train_raw['price_bucket'] = pd.qcut(np.log1p(train_raw['price']), q=10, labels=False, duplicates='drop')

train_part, temp_part = train_test_split(
    train_raw, test_size=0.30, stratify=train_raw['price_bucket'], random_state=42
)
val_part, test_part = train_test_split(
    temp_part, test_size=0.50, stratify=temp_part['price_bucket'], random_state=42
)

print("Train:", train_part.shape, "Val:", val_part.shape, "Test:", test_part.shape)

# sanity check
for name, split in [('train', train_part), ('val', val_part), ('test', test_part)]:
    print(name, "median:", split['price'].median(), "mean:", split['price'].mean())

Train: (52500, 5) Val: (11250, 5) Test: (11250, 5)
train median: 14.0 mean: 23.679899714285717
val median: 14.0 mean: 23.71024222222222
test median: 14.0 mean: 23.43458577777778


In [3]:
def smape(y_true, y_pred):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    denominator = (np.abs(y_true) + np.abs(y_pred)) / 2
    diff = np.abs(y_true - y_pred) / np.where(denominator == 0, 1, denominator)
    return 100 * np.mean(diff)

print(smape([10, 20, 30], [10, 20, 30]))  # 0
print(smape([10, 20, 30], [12, 18, 33]))  # small positive

0.0
12.743981165033796


In [4]:
def drop_derived_columns(df):
    derived_cols = [
        'text_len_chars', 'text_len_words', 'image_filename',
        'value', 'unit', 'has_description', 'has_bullets', 'has_value', 'num_bullets'
    ]
    existing = [c for c in derived_cols if c in df.columns]
    if existing:
        df = df.drop(columns=existing)
    return df

def extract_features(df, text_col='catalog_content', image_link_col='image_link'):
    df = drop_derived_columns(df.copy())
    text = df[text_col].astype(str)

    df['text_len_chars'] = text.apply(len)
    df['text_len_words'] = text.apply(lambda x: len(x.split()))

    if image_link_col in df.columns:
        df['image_filename'] = df[image_link_col].apply(lambda x: str(x).split('/')[-1])

    df['value'] = text.str.extract(r'Value:\s*([\d.]+)').astype(float)
    df['unit'] = text.str.extract(r'Unit:\s*([^\n]+)')[0].str.strip()

    df['num_bullets'] = text.apply(lambda t: len(re.findall(r'Bullet Point (\d+):', t)) or len(re.findall(r'Bullet Point:', t)))

    df['has_description'] = text.str.contains(r'Product Description:', na=False)
    df['has_bullets'] = df['num_bullets'] > 0
    df['has_value'] = df['value'].notna()

    return df

train_split = extract_features(train_part)
val_split = extract_features(val_part)
test_split = extract_features(test_part)
official_test = extract_features(official_test_raw)

print(train_split.shape)
train_split[['sample_id', 'value', 'unit', 'num_bullets', 'has_description', 'text_len_words']].head(3)


(52500, 14)


,sample_id,value,unit,num_bullets,has_description,text_len_words
66842,265060,3.00,Count,5,True,92
21484,118733,128.00,Fl Oz,5,True,168
30885,181454,2.01,Ounce,4,False,67


In [5]:
!pip install -q sentence-transformers

In [6]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("Torch version:", torch.__version__)

!nvidia-smi

CUDA available: True
Torch version: 2.10.0+cu128
Thu Aug  6 14:25:29 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.04             Driver Version: 580.159.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   42C    P8             10W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |

In [7]:
from sentence_transformers import SentenceTransformer

text_model = SentenceTransformer('all-MiniLM-L6-v2', device='cuda')
print(text_model)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

SentenceTransformer(
  (0): Transformer({'transformer_task': 'feature-extraction', 'modality_config': {'text': {'method': 'forward', 'method_output_name': 'last_hidden_state'}}, 'module_output_name': 'token_embeddings', 'architecture': 'BertModel'})
  (1): Pooling({'embedding_dimension': 384, 'pooling_mode': 'mean', 'include_prompt': True})
  (2): Normalize({})
)


In [8]:
import numpy as np

def get_text_embeddings(df, text_col='catalog_content', batch_size=128):
    texts = df[text_col].astype(str).tolist()
    embeddings = text_model.encode(
        texts,
        batch_size=batch_size,
        show_progress_bar=True,
        convert_to_numpy=True
    )
    return embeddings

train_text_emb = get_text_embeddings(train_split)
val_text_emb = get_text_embeddings(val_split)
test_text_emb = get_text_embeddings(test_split)
official_test_text_emb = get_text_embeddings(official_test)

print(train_text_emb.shape, val_text_emb.shape, test_text_emb.shape, official_test_text_emb.shape)

Batches:   0%|          | 0/411 [00:00<?, ?it/s]

Batches:   0%|          | 0/88 [00:00<?, ?it/s]

Batches:   0%|          | 0/88 [00:00<?, ?it/s]

Batches:   0%|          | 0/586 [00:00<?, ?it/s]

(52500, 384) (11250, 384) (11250, 384) (75000, 384)


In [9]:
np.save('/kaggle/working/train_text_emb.npy', train_text_emb)
np.save('/kaggle/working/val_text_emb.npy', val_text_emb)
np.save('/kaggle/working/test_text_emb.npy', test_text_emb)
np.save('/kaggle/working/official_test_text_emb.npy', official_test_text_emb)

In [10]:
!pip install -q git+https://github.com/openai/CLIP.git

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 1.1 MB/s eta 0:00:00


In [11]:
import clip
import torch
from PIL import Image

device = "cuda" if torch.cuda.is_available() else "cpu"
clip_model, clip_preprocess = clip.load("ViT-B/32", device=device)
print("Loaded CLIP on:", device)

100%|███████████████████████████████████████| 338M/338M [00:04<00:00, 80.6MiB/s]


Loaded CLIP on: cuda


In [12]:
import numpy as np
import os
from tqdm import tqdm

def get_image_embeddings(df, images_dir=IMAGES_DIR, filename_col='image_filename', batch_size=64):
    embeddings = []
    embed_dim = 512  # ViT-B/32 output dim

    filenames = df[filename_col].tolist()

    for i in tqdm(range(0, len(filenames), batch_size)):
        batch_files = filenames[i:i+batch_size]
        batch_imgs = []
        valid_indices = []

        for j, fname in enumerate(batch_files):
            path = os.path.join(images_dir, fname)
            try:
                img = Image.open(path).convert("RGB")
                batch_imgs.append(clip_preprocess(img))
                valid_indices.append(j)
            except Exception:
                pass  # missing/corrupt — will fill with zeros below

        batch_embeddings = np.zeros((len(batch_files), embed_dim), dtype=np.float32)

        if batch_imgs:
            batch_tensor = torch.stack(batch_imgs).to(device)
            with torch.no_grad():
                feats = clip_model.encode_image(batch_tensor)
            feats = feats.cpu().numpy()
            for idx, valid_idx in enumerate(valid_indices):
                batch_embeddings[valid_idx] = feats[idx]

        embeddings.append(batch_embeddings)

    return np.vstack(embeddings)

In [13]:
train_img_emb = get_image_embeddings(train_split)
val_img_emb = get_image_embeddings(val_split)
test_img_emb = get_image_embeddings(test_split)
official_test_img_emb = get_image_embeddings(official_test)



100%|██████████| 1172/1172 [06:15<00:00,  3.12it/s]


In [14]:
# Feature CSVs
train_split.to_csv('/kaggle/working/train_features.csv', index=False)
val_split.to_csv('/kaggle/working/val_features.csv', index=False)
test_split.to_csv('/kaggle/working/test_features.csv', index=False)
official_test.to_csv('/kaggle/working/official_test_features.csv', index=False)

# Text embeddings
np.save('/kaggle/working/train_text_emb.npy', train_text_emb)
np.save('/kaggle/working/val_text_emb.npy', val_text_emb)
np.save('/kaggle/working/test_text_emb.npy', test_text_emb)
np.save('/kaggle/working/official_test_text_emb.npy', official_test_text_emb)

# Image embeddings
np.save('/kaggle/working/train_img_emb.npy', train_img_emb)
np.save('/kaggle/working/val_img_emb.npy', val_img_emb)
np.save('/kaggle/working/test_img_emb.npy', test_img_emb)
np.save('/kaggle/working/official_test_img_emb.npy', official_test_img_emb)

print("All saved. Files:")
print(os.listdir('/kaggle/working'))

All saved. Files:
['val_img_emb.npy', 'train_img_emb.npy', 'official_test_img_emb.npy', 'train_features.csv', 'test_features.csv', 'val_features.csv', 'test_img_emb.npy', 'val_text_emb.npy', '__notebook__.ipynb', 'official_test_text_emb.npy', 'train_text_emb.npy', 'official_test_features.csv', 'test_text_emb.npy']
